In [ ]:
import os
import json
import yaml
import random
import re
import hashlib
import requests
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
SPEC_DIR = "data/openapi_specs"
OUTPUT = "data/generated/openapi_inventory.csv"

APIS_GURU_INDEX = "https://api.apis.guru/v2/list.json"
SAMPLE_SIZE = 45
SEED = 42

BANKING_SPECS = {
    "stripe.json": "https://raw.githubusercontent.com/stripe/openapi/master/openapi/spec3.json",
    "plaid.yaml": "https://raw.githubusercontent.com/plaid/plaid-openapi/master/2020-09-14.yml",
    "adyen_checkout.json": "https://raw.githubusercontent.com/Adyen/adyen-openapi/main/json/CheckoutService-v71.json",
    "adyen_payments.json": "https://raw.githubusercontent.com/Adyen/adyen-openapi/main/json/PaymentService-v68.json",
    "paypal_orders.json": "https://raw.githubusercontent.com/paypal/paypal-rest-api-specifications/main/openapi/checkout_orders_v2.json",
}

In [ ]:
Path(SPEC_DIR).mkdir(parents=True, exist_ok=True)

random.seed(SEED)
index = requests.get(APIS_GURU_INDEX, timeout=30).json()

candidates = []
for provider_key, provider in index.items():
    versions = provider.get("versions", {})
    if not versions:
        continue
    preferred = provider.get("preferred") or next(iter(versions))
    version_info = versions[preferred]
    url = version_info.get("swaggerUrl") or version_info.get("swaggerYamlUrl")
    if url:
        candidates.append((provider_key, url))

sampled = random.sample(candidates, min(SAMPLE_SIZE, len(candidates)))

for provider_key, url in sampled:
    suffix = Path(url).suffix or ".json"
    filename = provider_key.replace(":", "_").replace("/", "_") + suffix
    out_path = Path(SPEC_DIR) / filename
    if out_path.exists():
        continue
    response = requests.get(url, timeout=30)
    out_path.write_bytes(response.content)

print(f"APIs.guru sample: {len(sampled)} specs in {SPEC_DIR}")

In [ ]:
Path(SPEC_DIR).mkdir(parents=True, exist_ok=True)

for filename, url in BANKING_SPECS.items():
    out_path = Path(SPEC_DIR) / filename
    if out_path.exists():
        continue
    response = requests.get(url, timeout=60)
    out_path.write_bytes(response.content)

print(f"Banking specs: {len(BANKING_SPECS)} files in {SPEC_DIR}")

In [ ]:
records = []
spec_dir = Path(SPEC_DIR)

HTTP_METHODS = {"get", "post", "put", "patch", "delete", "head", "options", "trace"}


def resolve_auth_schemes(spec):
    schemes = (
        spec.get("components", {}).get("securitySchemes")
        or spec.get("securityDefinitions")
        or {}
    )
    return {
        name: (defn.get("type", "unknown"), defn.get("scheme", ""), defn.get("flow", ""))
        for name, defn in schemes.items()
    }


def auth_scheme_for(security_block, schemes_map):
    if not security_block:
        return "none"
    names = []
    for requirement in security_block:
        if not requirement:
            names.append("none")
            continue
        for scheme_name in requirement.keys():
            info = schemes_map.get(scheme_name)
            if not info:
                names.append(scheme_name)
                continue
            type_, sub, flow = info
            label = type_
            if sub:
                label = f"{type_}:{sub}"
            elif flow:
                label = f"{type_}:{flow}"
            names.append(label)
    return "|".join(sorted(set(names))) or "none"


def version_from_path(endpoint):
    m = re.search(r"/v(\d+(?:\.\d+)?)", endpoint)
    return m.group(1) if m else ""


def collect_schema_refs(details):
    refs = set()
    body = details.get("requestBody", {}) or {}
    for media in (body.get("content") or {}).values():
        schema = media.get("schema") or {}
        ref = schema.get("$ref")
        if ref:
            refs.add(ref.rsplit("/", 1)[-1])
    for param in details.get("parameters", []) or []:
        schema = param.get("schema") or {}
        ref = schema.get("$ref")
        if ref:
            refs.add(ref.rsplit("/", 1)[-1])
    for resp in (details.get("responses") or {}).values():
        if not isinstance(resp, dict):
            continue
        for media in (resp.get("content") or {}).values():
            schema = media.get("schema") or {}
            ref = schema.get("$ref")
            if ref:
                refs.add(ref.rsplit("/", 1)[-1])
        schema = resp.get("schema") or {}
        ref = schema.get("$ref")
        if ref:
            refs.add(ref.rsplit("/", 1)[-1])
    return sorted(refs)


for file in spec_dir.rglob("*"):
    if file.suffix == ".json":
        spec = json.loads(file.read_text())
    elif file.suffix in [".yaml", ".yml"]:
        spec = yaml.safe_load(file.read_text())
    else:
        continue

    info = spec.get("info", {}) or {}
    spec_version = info.get("version", "")
    service = info.get("x-serviceName") or info.get("title", "")
    provider = info.get("x-providerName", "")
    schemes_map = resolve_auth_schemes(spec)
    global_security = spec.get("security", [])

    paths = spec.get("paths", {}) or {}
    for endpoint, methods in paths.items():
        for method, details in methods.items():
            if method.lower() not in HTTP_METHODS:
                continue

            security = details.get("security", global_security)
            auth_scheme = auth_scheme_for(security, schemes_map)

            tags = details.get("tags", []) or []
            schemas = collect_schema_refs(details)

            records.append({
                "source_file": file.name,
                "endpoint": endpoint,
                "method": method.upper(),
                "version_path": version_from_path(endpoint),
                "version_spec": spec_version,
                "service": service,
                "provider": provider,
                "tags": "|".join(tags),
                "deprecated": int(details.get("deprecated", False)),
                "auth_scheme": auth_scheme,
                "schemas": "|".join(schemas),
                "schema_count": len(schemas),
                "operation_id": details.get("operationId", ""),
            })

df = pd.DataFrame(records)

os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
df.to_csv(OUTPUT, index=False)

print(df.shape)
df.head()

In [ ]:
BANKING_OVERLAY_OUT = "data/generated/banking_inventory.csv"

BANKING_PATHS = [
    "/v1/accounts/{accountId}/balance",
    "/v1/accounts/{accountId}/transactions",
    "/v1/accounts/{accountId}/statement",
    "/v1/accounts/{accountId}/holds",
    "/v1/upi/collect",
    "/v1/upi/pay",
    "/v1/upi/mandate",
    "/v1/upi/vpa/verify",
    "/v1/transfers/imps",
    "/v1/transfers/neft",
    "/v1/transfers/rtgs",
    "/v1/transfers/swift",
    "/v1/cards/{cardId}/freeze",
    "/v1/cards/{cardId}/limit",
    "/v1/cards/{cardId}/pin",
    "/v1/cards/issue",
    "/v1/loans/{loanId}/repayment",
    "/v1/loans/{loanId}/schedule",
    "/v1/loans/apply",
    "/v1/loans/eligibility",
    "/v1/kyc/verify",
    "/v1/kyc/aadhaar",
    "/v1/kyc/pan",
    "/v1/kyc/video",
    "/v1/customers/{customerId}/profile",
    "/v1/customers/{customerId}/documents",
    "/v1/customers/{customerId}/preferences",
    "/v1/beneficiaries",
    "/v1/beneficiaries/{beneficiaryId}",
    "/v1/aa/consent",
    "/v1/aa/data",
    "/v1/aa/fip/discover",
    "/v1/fd/book",
    "/v1/fd/{depositId}/close",
    "/v1/rd/book",
    "/v1/forex/quote",
    "/v1/forex/book",
    "/v1/notifications/preferences",
    "/v1/audit/logs",
    "/v2/accounts/{accountId}/balance",
    "/v2/transfers/imps",
    "/v2/upi/pay",
    "/v2/cards/{cardId}/freeze",
    "/v3/accounts/{accountId}/balance",
    "/v3/upi/pay",
]

BANKING_SERVICES = [
    "core-banking",
    "payments",
    "cards",
    "loans",
    "kyc",
    "wealth",
    "forex",
    "notifications",
    "audit",
    "aa-bridge",
]


def stable_hash(s, mod):
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % mod


inventory = pd.read_csv(OUTPUT)

rows = []
for endpoint_id, (_, row) in enumerate(inventory.iterrows()):
    key = f"{row['source_file']}|{row['endpoint']}|{row['method']}"
    banking_path = BANKING_PATHS[stable_hash(key, len(BANKING_PATHS))]
    service = BANKING_SERVICES[stable_hash(key + "svc", len(BANKING_SERVICES))]
    version_match = re.search(r"/v(\d+)", banking_path)
    version_path = version_match.group(1) if version_match else ""

    rows.append({
        "endpoint_id": endpoint_id,
        "endpoint": banking_path,
        "method": row["method"],
        "service": service,
        "version_path": version_path,
        "auth_scheme": row["auth_scheme"],
        "schema_count": row["schema_count"],
        "deprecated_flag": row["deprecated"],
        "tags": row["tags"] if pd.notna(row["tags"]) else "",
        "seed_source": row["source_file"],
    })

banking_df = pd.DataFrame(rows)
banking_df.to_csv(BANKING_OVERLAY_OUT, index=False)

print(banking_df.shape)
print("unique endpoint_id:", banking_df["endpoint_id"].nunique(), "of", len(banking_df))
print("unique banking endpoints:", banking_df["endpoint"].nunique())
print("services:", banking_df["service"].value_counts().to_dict())
banking_df.head()

In [ ]:
CVE_TABLE = "data/generated/cve_table.csv"

CVE_DATA = [
    {"runtime": "springboot", "version": "2.5.0", "cve_id": "CVE-2022-22965", "cvss": 9.8, "summary": "Spring4Shell RCE via data binding"},
    {"runtime": "springboot", "version": "2.7.18", "cve_id": "CVE-2023-20860", "cvss": 7.5, "summary": "Security bypass via pattern matching"},
    {"runtime": "springboot", "version": "3.2.0", "cve_id": "CVE-2024-22243", "cvss": 6.1, "summary": "URL parsing open redirect"},
    {"runtime": "nodejs", "version": "14.21", "cve_id": "CVE-2023-30581", "cvss": 8.8, "summary": "Permission model bypass via Module"},
    {"runtime": "nodejs", "version": "18.19", "cve_id": "CVE-2024-22019", "cvss": 7.5, "summary": "HTTP request smuggling chunked encoding"},
    {"runtime": "nodejs", "version": "20.11", "cve_id": "CVE-2024-21892", "cvss": 7.8, "summary": "Privilege escalation via env vars"},
    {"runtime": "python", "version": "3.7", "cve_id": "CVE-2023-24329", "cvss": 7.5, "summary": "urllib.parse scheme bypass"},
    {"runtime": "python", "version": "3.9", "cve_id": "CVE-2023-40217", "cvss": 5.3, "summary": "TLS handshake data leak"},
    {"runtime": "python", "version": "3.11", "cve_id": "CVE-2024-0397", "cvss": 5.9, "summary": "ssl module memory race"},
    {"runtime": "golang", "version": "1.19", "cve_id": "CVE-2023-29406", "cvss": 6.5, "summary": "HTTP host header validation bypass"},
    {"runtime": "golang", "version": "1.21", "cve_id": "CVE-2024-24784", "cvss": 6.5, "summary": "net/mail comment parsing"},
    {"runtime": "dotnet", "version": "6.0", "cve_id": "CVE-2024-21386", "cvss": 7.5, "summary": "SignalR denial of service"},
    {"runtime": "dotnet", "version": "8.0", "cve_id": "CVE-2024-30045", "cvss": 6.8, "summary": "Remote code execution via deserialization"},
]

os.makedirs(os.path.dirname(CVE_TABLE), exist_ok=True)
cve_df = pd.DataFrame(CVE_DATA)
cve_df.to_csv(CVE_TABLE, index=False)

print(cve_df.shape)
cve_df

In [ ]:
LIFECYCLE_OUT = "data/generated/lifecycle_training.csv"

RUNTIMES = [
    ("springboot", "2.5.0"),
    ("springboot", "2.7.18"),
    ("springboot", "3.2.0"),
    ("nodejs", "14.21"),
    ("nodejs", "18.19"),
    ("nodejs", "20.11"),
    ("python", "3.7"),
    ("python", "3.9"),
    ("python", "3.11"),
    ("golang", "1.19"),
    ("golang", "1.21"),
    ("dotnet", "6.0"),
    ("dotnet", "8.0"),
]

SCENARIO_WEIGHTS = {
    "active": 0.50,
    "deprecated": 0.15,
    "orphaned_quiet": 0.10,
    "zombie": 0.15,
    "shadow": 0.10,
}

SCENARIO_TO_LABEL = {
    "active":         ("active",     0, 0),
    "deprecated":     ("deprecated", 0, 0),
    "orphaned_quiet": ("orphaned",   0, 0),
    "zombie":         ("orphaned",   1, 0),
    "shadow":         ("orphaned",   0, 1),
}

ACTIVE_AUTH_FALLBACKS = ["apiKey", "oauth2:implicit", "http:bearer", "http:basic|http:bearer"]
BOUNDARY_RATE = 0.12

random.seed(SEED)
np.random.seed(SEED)

banking_df = pd.read_csv(BANKING_OVERLAY_OUT)
cve_df = pd.read_csv(CVE_TABLE)
cve_lookup = {(r["runtime"], r["version"]): r for _, r in cve_df.iterrows()}

scenario_names = list(SCENARIO_WEIGHTS.keys())
scenario_weights = list(SCENARIO_WEIGHTS.values())


def features_for_scenario(scenario):
    if scenario == "active":
        return {
            "in_registry": 1,
            "last_seen_days": random.randint(0, 2),
            "call_count_7d": random.randint(500, 50000),
            "auth_fail_rate_7d": round(random.uniform(0.0, 0.02), 4),
            "p95_latency_ms": random.randint(40, 400),
            "last_deploy_days": random.randint(0, 60),
            "owner_present": 1,
        }
    if scenario == "deprecated":
        return {
            "in_registry": 1,
            "last_seen_days": random.randint(0, 14),
            "call_count_7d": random.randint(10, 500),
            "auth_fail_rate_7d": round(random.uniform(0.0, 0.05), 4),
            "p95_latency_ms": random.randint(100, 800),
            "last_deploy_days": random.randint(180, 720),
            "owner_present": 1,
        }
    if scenario == "orphaned_quiet":
        return {
            "in_registry": 1,
            "last_seen_days": random.randint(7, 90),
            "call_count_7d": random.randint(0, 100),
            "auth_fail_rate_7d": round(random.uniform(0.05, 0.30), 4),
            "p95_latency_ms": random.randint(200, 2000),
            "last_deploy_days": random.randint(365, 1500),
            "owner_present": 0,
        }
    if scenario == "zombie":
        return {
            "in_registry": 1,
            "last_seen_days": random.randint(0, 2),
            "call_count_7d": random.randint(3000, 30000),
            "auth_fail_rate_7d": round(random.uniform(0.02, 0.10), 4),
            "p95_latency_ms": random.randint(200, 1500),
            "last_deploy_days": random.randint(365, 1500),
            "owner_present": 0,
        }
    return {
        "in_registry": 0,
        "last_seen_days": random.randint(0, 7),
        "call_count_7d": random.randint(50, 5000),
        "auth_fail_rate_7d": round(random.uniform(0.10, 0.50), 4),
        "p95_latency_ms": random.randint(50, 1500),
        "last_deploy_days": random.randint(0, 90),
        "owner_present": 0,
    }


def add_boundary_noise(scenario, features):
    if random.random() >= BOUNDARY_RATE:
        return features
    f = dict(features)
    if scenario == "active":
        c = random.choice(["low_traffic", "old_deploy", "lost_owner", "fail_spike"])
        if c == "low_traffic":
            f["call_count_7d"] = random.randint(10, 500)
        elif c == "old_deploy":
            f["last_deploy_days"] = random.randint(180, 720)
        elif c == "lost_owner":
            f["owner_present"] = 0
        else:
            f["auth_fail_rate_7d"] = round(random.uniform(0.05, 0.20), 4)
    elif scenario == "deprecated":
        c = random.choice(["high_traffic", "no_owner", "fresh_deploy"])
        if c == "high_traffic":
            f["call_count_7d"] = random.randint(2000, 15000)
        elif c == "no_owner":
            f["owner_present"] = 0
        else:
            f["last_deploy_days"] = random.randint(0, 90)
    elif scenario == "orphaned_quiet":
        c = random.choice(["claimed_owner", "recent_deploy", "low_fail"])
        if c == "claimed_owner":
            f["owner_present"] = 1
        elif c == "recent_deploy":
            f["last_deploy_days"] = random.randint(0, 90)
        else:
            f["auth_fail_rate_7d"] = round(random.uniform(0.0, 0.05), 4)
    elif scenario == "zombie":
        c = random.choice(["claimed_owner", "lower_traffic", "fresh_deploy"])
        if c == "claimed_owner":
            f["owner_present"] = 1
        elif c == "lower_traffic":
            f["call_count_7d"] = random.randint(50, 2999)
        else:
            f["last_deploy_days"] = random.randint(0, 90)
    else:
        c = random.choice(["registered_now", "claimed_owner"])
        if c == "registered_now":
            f["in_registry"] = 1
        else:
            f["owner_present"] = 1
    return f


def compute_risk(row, max_cvss):
    base = 0.0
    base += 30 * (1 - row["in_registry"])
    base += 40 * row["auth_fail_rate_7d"]
    base += 20 * (max_cvss / 10.0)
    base += 15 * (1 - row["owner_present"])
    base += 10 * min(row["last_deploy_days"] / 1000.0, 1.0)
    base += 15 if row["auth_scheme"] == "none" else 0
    if row["is_zombie"] == 1:
        base += 20
    base += np.random.normal(0, 5)
    return max(0.0, min(100.0, base))


records = []
for _, seed_row in banking_df.iterrows():
    scenario = random.choices(scenario_names, weights=scenario_weights)[0]
    lifecycle_state, is_zombie, is_shadow = SCENARIO_TO_LABEL[scenario]
    features = features_for_scenario(scenario)
    features = add_boundary_noise(scenario, features)
    runtime, runtime_version = RUNTIMES[random.randint(0, len(RUNTIMES) - 1)]
    cve = cve_lookup.get((runtime, runtime_version))
    cve_id = cve["cve_id"] if cve is not None else ""
    max_cvss = float(cve["cvss"]) if cve is not None else 0.0

    if scenario == "shadow":
        endpoint = re.sub(r"/v\d+/", "/internal/", seed_row["endpoint"])
        auth_scheme = "none" if random.random() < 0.6 else seed_row["auth_scheme"]
        schema_count = 0
        deprecated_flag = 0
    elif scenario == "zombie":
        endpoint = seed_row["endpoint"]
        auth_scheme = seed_row["auth_scheme"]
        schema_count = int(seed_row["schema_count"])
        deprecated_flag = 1 if random.random() < 0.6 else int(seed_row["deprecated_flag"])
    else:
        endpoint = seed_row["endpoint"]
        auth_scheme = seed_row["auth_scheme"]
        schema_count = int(seed_row["schema_count"])
        deprecated_flag = 1 if scenario == "deprecated" else int(seed_row["deprecated_flag"])

    if scenario == "active" and auth_scheme == "none":
        auth_scheme = random.choice(ACTIVE_AUTH_FALLBACKS)

    row = {
        "endpoint_id": int(seed_row["endpoint_id"]),
        "endpoint": endpoint,
        "method": seed_row["method"],
        "service": seed_row["service"],
        "version_path": seed_row["version_path"],
        "auth_scheme": auth_scheme,
        "schema_count": schema_count,
        "deprecated_flag": deprecated_flag,
        **features,
        "runtime": runtime,
        "runtime_version": runtime_version,
        "cve_id": cve_id,
        "max_cvss": max_cvss,
        "lifecycle_state": lifecycle_state,
        "is_zombie": is_zombie,
        "is_shadow": is_shadow,
    }
    row["risk_score"] = round(compute_risk(row, max_cvss), 2)
    records.append(row)

lifecycle_df = pd.DataFrame(records)
lifecycle_df.to_csv(LIFECYCLE_OUT, index=False)

print(lifecycle_df.shape)
print("lifecycle_state:")
print(lifecycle_df["lifecycle_state"].value_counts())
print("is_zombie:", int(lifecycle_df["is_zombie"].sum()))
print("is_shadow:", int(lifecycle_df["is_shadow"].sum()))
print(f"boundary noise rate: {BOUNDARY_RATE} (rows where features deliberately overlap neighbor class)")
print("risk_score range:", lifecycle_df["risk_score"].min(), "-", lifecycle_df["risk_score"].max())
print("critical (>=90):", int((lifecycle_df["risk_score"] >= 90).sum()))
print("high (75-90):", int(((lifecycle_df["risk_score"] >= 75) & (lifecycle_df["risk_score"] < 90)).sum()))
lifecycle_df.head()

In [ ]:
SEQUENCE_OUT = "data/generated/lifecycle_sequences.csv"
SEQUENCE_DAYS = 30
ANOMALY_RATE = 0.10

random.seed(SEED + 1)
np.random.seed(SEED + 1)

lifecycle_df = pd.read_csv(LIFECYCLE_OUT)


def derive_scenario(row):
    if row["is_shadow"] == 1:
        return "shadow"
    if row["is_zombie"] == 1:
        return "zombie"
    if row["lifecycle_state"] == "active":
        return "active"
    if row["lifecycle_state"] == "deprecated":
        return "deprecated"
    return "orphaned_quiet"


def sequence_for(scenario, days):
    if scenario == "active":
        base = np.random.randint(800, 5000)
        calls = base + np.random.normal(0, base * 0.05, days)
        auth_fail = np.random.normal(0.01, 0.005, days)
        latency = np.random.normal(120, 15, days)
    elif scenario == "deprecated":
        base = np.random.randint(200, 1500)
        calls = base * np.linspace(1.0, 0.2, days) + np.random.normal(0, base * 0.05, days)
        auth_fail = np.random.normal(0.03, 0.01, days)
        latency = np.random.normal(250, 40, days)
    elif scenario == "orphaned_quiet":
        base = np.random.randint(0, 80)
        calls = base + np.random.normal(0, 10, days)
        auth_fail = np.random.normal(0.15, 0.05, days)
        latency = np.random.normal(600, 200, days)
    elif scenario == "zombie":
        base = np.random.randint(3000, 15000)
        calls = base + np.random.normal(0, base * 0.08, days)
        auth_fail = np.random.normal(0.05, 0.02, days)
        latency = np.random.normal(450, 100, days)
    else:
        base = np.random.randint(100, 2000)
        calls = base + np.random.normal(0, base * 0.4, days)
        auth_fail = np.random.normal(0.25, 0.10, days)
        latency = np.random.normal(400, 200, days)

    if random.random() < ANOMALY_RATE:
        shift = random.randint(days // 3, 2 * days // 3)
        calls[shift:] = calls[shift:] * random.choice([0.1, 3.0])
        anomaly = 1
    else:
        anomaly = 0

    return (
        np.clip(calls, 0, None),
        np.clip(auth_fail, 0, 1),
        np.clip(latency, 10, None),
        anomaly,
    )


rows = []
for _, ep in lifecycle_df.iterrows():
    scenario = derive_scenario(ep)
    calls, auth_fail, latency, anomaly = sequence_for(scenario, SEQUENCE_DAYS)
    for day in range(SEQUENCE_DAYS):
        rows.append({
            "endpoint_id": int(ep["endpoint_id"]),
            "endpoint": ep["endpoint"],
            "method": ep["method"],
            "day": day,
            "call_count": int(calls[day]),
            "auth_fail_rate": round(float(auth_fail[day]), 4),
            "p95_latency_ms": round(float(latency[day]), 1),
            "lifecycle_state": ep["lifecycle_state"],
            "is_zombie": int(ep["is_zombie"]),
            "is_shadow": int(ep["is_shadow"]),
            "anomaly": anomaly,
        })

seq_df = pd.DataFrame(rows)
seq_df.to_csv(SEQUENCE_OUT, index=False)

print(seq_df.shape)
print("endpoints:", seq_df["endpoint_id"].nunique())
print("anomaly endpoints:", seq_df.groupby("endpoint_id")["anomaly"].first().sum())
seq_df.head()

In [ ]:
inv = pd.read_csv(OUTPUT)

print("shape:", inv.shape)
print("specs ingested:", inv["source_file"].nunique())
print()
print("top 10 sources:")
print(inv["source_file"].value_counts().head(10))
print()
print("methods:")
print(inv["method"].value_counts())
print()
print("auth schemes:")
print(inv["auth_scheme"].value_counts())
print()
print("deprecated:", int(inv["deprecated"].sum()), "/", len(inv))
print()
print("schema_count stats:")
print(inv["schema_count"].describe().round(2))
print()
print("nulls:")
print(inv.isna().sum())

In [ ]:
bnk = pd.read_csv(BANKING_OVERLAY_OUT)

print("shape:", bnk.shape)
print("unique endpoints:", bnk["endpoint"].nunique())
print()
print("service distribution:")
print(bnk["service"].value_counts())
print()
print("version_path distribution:")
print(bnk["version_path"].value_counts())
print()
print("auth_scheme distribution (preserved from source):")
print(bnk["auth_scheme"].value_counts().head(10))
print()
print("rows per endpoint top 5:")
print(bnk["endpoint"].value_counts().head(5))
print()
print("rows per endpoint bottom 5:")
print(bnk["endpoint"].value_counts().tail(5))

In [ ]:
life = pd.read_csv(LIFECYCLE_OUT)

print("shape:", life.shape)
print()
print("lifecycle_state (3-class):")
print(life["lifecycle_state"].value_counts())
print(life["lifecycle_state"].value_counts(normalize=True).round(3))
print()
print("flags:")
print("  is_zombie:", int(life["is_zombie"].sum()))
print("  is_shadow:", int(life["is_shadow"].sum()))
print()

print("sanity checks:")
print("  is_shadow=1 → in_registry unique:", life[life["is_shadow"] == 1]["in_registry"].unique())
print("  is_zombie=1 → owner_present unique:", life[life["is_zombie"] == 1]["owner_present"].unique())
print("  is_zombie=1 → call_count_7d min:", int(life[life["is_zombie"] == 1]["call_count_7d"].min()))
print("  active → owner_present unique:", life[life["lifecycle_state"] == "active"]["owner_present"].unique())
print("  active → auth=none count:", int((life[life["lifecycle_state"] == "active"]["auth_scheme"] == "none").sum()))
print()

feature_cols = [
    "in_registry", "last_seen_days", "call_count_7d", "auth_fail_rate_7d",
    "p95_latency_ms", "last_deploy_days", "owner_present", "schema_count",
    "max_cvss", "risk_score",
]
print("feature means by lifecycle_state:")
print(life.groupby("lifecycle_state")[feature_cols].mean().round(2))
print()

print("feature means by sub-population:")
sub = life.copy()
sub["bucket"] = sub.apply(
    lambda r: "shadow" if r["is_shadow"] else ("zombie" if r["is_zombie"] else r["lifecycle_state"]),
    axis=1,
)
print(sub.groupby("bucket")[["call_count_7d", "owner_present", "in_registry", "risk_score"]].mean().round(2))
print()

print("risk_score buckets:")
print("  critical (>=90):", int((life["risk_score"] >= 90).sum()))
print("  high (75-90):   ", int(((life["risk_score"] >= 75) & (life["risk_score"] < 90)).sum()))
print("  medium (40-75): ", int(((life["risk_score"] >= 40) & (life["risk_score"] < 75)).sum()))
print("  low (<40):      ", int((life["risk_score"] < 40).sum()))
print()

print("risk_score by lifecycle_state:")
print(life.groupby("lifecycle_state")["risk_score"].describe()[["min", "mean", "max", "std"]].round(2))
print()
print("CVE coverage:", (life["max_cvss"] > 0).sum(), "/", len(life))

In [ ]:
seq = pd.read_csv(SEQUENCE_OUT)

print("shape:", seq.shape)
print("endpoints:", seq["endpoint_id"].nunique())
print("days per endpoint:", seq["day"].nunique())
print()

ep_summary = seq.groupby("endpoint_id").agg(state=("lifecycle_state", "first"), anomaly=("anomaly", "first"))
print("anomaly endpoint count by state:")
print(ep_summary.groupby("state")["anomaly"].sum())
print()
print("anomaly rate by state:")
print(ep_summary.groupby("state")["anomaly"].mean().round(3))
print()

print("call_count trend: first 5d mean vs last 5d mean by state")
first = seq[seq["day"] < 5].groupby("lifecycle_state")["call_count"].mean()
last = seq[seq["day"] >= 25].groupby("lifecycle_state")["call_count"].mean()
trend = pd.DataFrame({"first_5d": first, "last_5d": last})
trend["delta_pct"] = ((trend["last_5d"] - trend["first_5d"]) / trend["first_5d"].replace(0, 1) * 100).round(1)
print(trend.round(1))
print()

print("auth_fail_rate mean by state:")
print(seq.groupby("lifecycle_state")["auth_fail_rate"].mean().round(4))
print()
print("p95_latency mean by state:")
print(seq.groupby("lifecycle_state")["p95_latency_ms"].mean().round(1))

In [ ]:
FINDINGS_OUT = "data/generated/lifecycle_findings.csv"

WEAK_AUTH = {"none", "apiKey", "basic", "apiKey|basic"}


def owasp_findings(row):
    findings = []

    if row["auth_scheme"] == "none" or row["auth_fail_rate_7d"] > 0.10:
        findings.append("API2:Broken-Authentication")

    if row["max_cvss"] >= 7.0:
        findings.append("API8:Security-Misconfiguration")

    if row["in_registry"] == 0:
        findings.append("API9:Improper-Inventory-Management")
    elif row["deprecated_flag"] == 1 and row["call_count_7d"] > 0:
        findings.append("API9:Improper-Inventory-Management")
    elif row["last_deploy_days"] > 365 and row["owner_present"] == 0:
        findings.append("API9:Improper-Inventory-Management")

    if re.search(r"\{\w*[Ii]d\}", row["endpoint"]) and row["auth_scheme"] in WEAK_AUTH:
        findings.append("API1:BOLA")

    if row["p95_latency_ms"] > 1000:
        findings.append("API4:Unrestricted-Resource-Consumption")

    if row["last_deploy_days"] > 720 and row["max_cvss"] >= 7.0:
        findings.append("API10:Unsafe-Consumption-Of-APIs")

    return findings


life = pd.read_csv(LIFECYCLE_OUT)

life["owasp_findings"] = life.apply(lambda r: "|".join(owasp_findings(r)), axis=1)
life["finding_count"] = life["owasp_findings"].apply(lambda s: 0 if s == "" else len(s.split("|")))

life.to_csv(FINDINGS_OUT, index=False)

print("shape:", life.shape)
print()
print("finding_count distribution:")
print(life["finding_count"].value_counts().sort_index())
print()
print("findings by category (multilabel):")
exploded = life["owasp_findings"].str.split("|").explode()
print(exploded[exploded != ""].value_counts())
print()
print("mean findings by lifecycle_state:")
print(life.groupby("lifecycle_state")["finding_count"].mean().round(2))
print()
print("mean findings by zombie/shadow flags:")
print("  is_zombie=1:", round(life[life["is_zombie"] == 1]["finding_count"].mean(), 2))
print("  is_shadow=1:", round(life[life["is_shadow"] == 1]["finding_count"].mean(), 2))
print()
print("sample zombie endpoints:")
zombies = life[life["is_zombie"] == 1]
if len(zombies) > 0:
    print(zombies[["endpoint_id", "endpoint", "method", "risk_score", "owasp_findings"]].sample(min(5, len(zombies)), random_state=SEED).to_string(index=False))

In [ ]:
PATHS = {
    "inventory": OUTPUT,
    "banking": BANKING_OVERLAY_OUT,
    "cve": CVE_TABLE,
    "lifecycle": LIFECYCLE_OUT,
    "sequences": SEQUENCE_OUT,
    "findings": FINDINGS_OUT,
}
dfs = {k: pd.read_csv(v) for k, v in PATHS.items()}

print("=== FILE SHAPES ===")
for k, df in dfs.items():
    print(f"  {k:12s}: {df.shape}")
print()

life = dfs["lifecycle"]
ids_b = set(dfs["banking"]["endpoint_id"])
ids_l = set(life["endpoint_id"])
ids_f = set(dfs["findings"]["endpoint_id"])
ids_s = set(dfs["sequences"]["endpoint_id"].unique())

print("=== JOIN KEY INTEGRITY ===")
print(f"  banking unique: {dfs['banking']['endpoint_id'].is_unique}")
print(f"  lifecycle unique: {life['endpoint_id'].is_unique}")
print(f"  findings unique: {dfs['findings']['endpoint_id'].is_unique}")
print(f"  same id set across all 4 files: {ids_b == ids_l == ids_f == ids_s}")
print(f"  sequence rows-per-endpoint: {dfs['sequences'].groupby('endpoint_id').size().unique()}")
print()

print("=== CLASS BALANCE ===")
print(life["lifecycle_state"].value_counts().to_string())
print(f"is_zombie={int(life['is_zombie'].sum())}  is_shadow={int(life['is_shadow'].sum())}  both_flags={int(((life['is_zombie']==1)&(life['is_shadow']==1)).sum())}")
print()

print("=== SANITY ===")
a = life[life["lifecycle_state"] == "active"]
o = life[life["lifecycle_state"] == "orphaned"]
sh = life[life["is_shadow"] == 1]
z = life[life["is_zombie"] == 1]
print(f"  active w/ in_registry=0: {int((a['in_registry']==0).sum())}  (expect 0)")
print(f"  active w/ owner_present=0: {int((a['owner_present']==0).sum())}  (expect 0)")
print(f"  active w/ auth=none: {int((a['auth_scheme']=='none').sum())}  (expect 0)")
print(f"  orphaned w/ owner_present=1: {int((o['owner_present']==1).sum())}  (expect 0)")
print(f"  is_shadow=1 w/ in_registry=1: {int((sh['in_registry']==1).sum())}  (expect 0)")
print(f"  is_zombie=1 w/ owner_present=1: {int((z['owner_present']==1).sum())}  (expect 0)")
print(f"  is_zombie=1 w/ call_count_7d<3000: {int((z['call_count_7d']<3000).sum())}  (expect 0)")
print()

print("=== RISK BANDS ===")
print(f"  critical (>=90): {int((life['risk_score']>=90).sum())}")
print(f"  high     (75-90): {int(((life['risk_score']>=75)&(life['risk_score']<90)).sum())}")
print(f"  medium   (40-75): {int(((life['risk_score']>=40)&(life['risk_score']<75)).sum())}")
print(f"  low      (<40):   {int((life['risk_score']<40).sum())}")
print()

print("=== NULLS ===")
for k in ["lifecycle", "findings", "sequences", "banking"]:
    nulls = dfs[k].isna().sum()
    nz = {c: int(v) for c, v in nulls.items() if v > 0}
    print(f"  {k}: {nz or 'none'}")

In [ ]:
life = pd.read_csv(LIFECYCLE_OUT)
sub = life.copy()
sub["bucket"] = sub.apply(lambda r: "shadow" if r["is_shadow"] else ("zombie" if r["is_zombie"] else r["lifecycle_state"]), axis=1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

states = life["lifecycle_state"].value_counts().reindex(["active", "deprecated", "orphaned"])
ax[0].bar(states.index, states.values, color=["#2ca02c", "#ff7f0e", "#9467bd"])
ax[0].set_title("lifecycle_state (model 1 label, 3-class)")
ax[0].set_ylabel("count")
for i, v in enumerate(states.values):
    ax[0].text(i, v + 15, str(int(v)), ha="center", fontweight="bold")

order = ["active", "deprecated", "orphaned", "zombie", "shadow"]
colors = ["#2ca02c", "#ff7f0e", "#9467bd", "#d62728", "#7f7f7f"]
bucket_counts = sub["bucket"].value_counts().reindex(order)
ax[1].bar(bucket_counts.index, bucket_counts.values, color=colors)
ax[1].set_title("sub-populations (lifecycle + flag overlays)")
ax[1].set_ylabel("count")
for i, v in enumerate(bucket_counts.values):
    ax[1].text(i, v + 10, str(int(v)), ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
life = pd.read_csv(LIFECYCLE_OUT)

fig, ax = plt.subplots(figsize=(11, 4))
n, bins, patches = ax.hist(life["risk_score"], bins=40, color="#1f77b4", edgecolor="white")
ax.axvspan(0, 40, alpha=0.10, color="green")
ax.axvspan(40, 75, alpha=0.10, color="gold")
ax.axvspan(75, 90, alpha=0.15, color="orange")
ax.axvspan(90, 100, alpha=0.20, color="red")

y_top = max(n) * 1.05
ax.text(20, y_top, "LOW", ha="center", fontweight="bold", fontsize=10)
ax.text(57, y_top, "MEDIUM", ha="center", fontweight="bold", fontsize=10)
ax.text(82, y_top, "HIGH", ha="center", fontweight="bold", fontsize=10)
ax.text(95, y_top, "CRITICAL", ha="center", fontweight="bold", fontsize=10, color="darkred")

ax.set_xlabel("risk_score")
ax.set_ylabel("endpoint count")
ax.set_title(f"Risk Score Distribution (n={len(life)})")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
life = pd.read_csv(LIFECYCLE_OUT)
sub = life.copy()
sub["bucket"] = sub.apply(lambda r: "shadow" if r["is_shadow"] else ("zombie" if r["is_zombie"] else r["lifecycle_state"]), axis=1)

order = ["active", "deprecated", "orphaned", "zombie", "shadow"]
colors = ["#2ca02c", "#ff7f0e", "#9467bd", "#d62728", "#7f7f7f"]
data = [sub[sub["bucket"] == o]["risk_score"].values for o in order]

fig, ax = plt.subplots(figsize=(11, 4.5))
bp = ax.boxplot(data, tick_labels=order, patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.axhline(90, color="red", linestyle="--", alpha=0.6, label="critical (90)")
ax.axhline(75, color="orange", linestyle="--", alpha=0.6, label="high (75)")
ax.axhline(40, color="gold", linestyle="--", alpha=0.5, label="medium (40)")
ax.set_ylabel("risk_score")
ax.set_title("Risk Score by Sub-population")
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
life = pd.read_csv(LIFECYCLE_OUT)
sub = life.copy()
sub["bucket"] = sub.apply(lambda r: "shadow" if r["is_shadow"] else ("zombie" if r["is_zombie"] else r["lifecycle_state"]), axis=1)

order = ["active", "deprecated", "orphaned", "zombie", "shadow"]
features = ["call_count_7d", "auth_fail_rate_7d", "p95_latency_ms", "last_deploy_days", "owner_present", "max_cvss"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, feat in zip(axes.flat, features):
    data = [sub[sub["bucket"] == o][feat].values for o in order]
    ax.boxplot(data, tick_labels=order)
    ax.set_title(feat, fontweight="bold")
    ax.tick_params(axis="x", rotation=20)
    if feat == "call_count_7d":
        ax.set_yscale("symlog")

plt.suptitle("Feature distributions by sub-population (separability check)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
seq = pd.read_csv(SEQUENCE_OUT)
sub = seq.copy()
sub["bucket"] = sub.apply(lambda r: "shadow" if r["is_shadow"] else ("zombie" if r["is_zombie"] else r["lifecycle_state"]), axis=1)

order = ["active", "deprecated", "orphaned", "zombie", "shadow"]
colors = ["#2ca02c", "#ff7f0e", "#9467bd", "#d62728", "#7f7f7f"]

fig, ax = plt.subplots(figsize=(11, 5))
for bucket, color in zip(order, colors):
    mean_by_day = sub[(sub["bucket"] == bucket) & (sub["anomaly"] == 0)].groupby("day")["call_count"].mean()
    ax.plot(mean_by_day.index, mean_by_day.values, label=bucket, color=color, linewidth=2.5)

ax.set_xlabel("day")
ax.set_ylabel("mean call_count (log)")
ax.set_title("30-Day call_count Trajectory by Sub-population (anomaly-free)")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
seq = pd.read_csv(SEQUENCE_OUT)
anom_ids = sorted(seq[seq["anomaly"] == 1]["endpoint_id"].unique())[:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, ep_id in zip(axes.flat, anom_ids):
    ep_data = seq[seq["endpoint_id"] == ep_id].sort_values("day")
    state = ep_data["lifecycle_state"].iloc[0]
    is_z = int(ep_data["is_zombie"].iloc[0])
    is_s = int(ep_data["is_shadow"].iloc[0])
    badge = " [zombie]" if is_z else (" [shadow]" if is_s else "")
    ax.plot(ep_data["day"], ep_data["call_count"], color="#d62728", linewidth=2, marker="o", markersize=3)
    ax.set_title(f"id={ep_id} | {state}{badge}", fontsize=10)
    ax.set_xlabel("day")
    ax.set_ylabel("call_count")
    ax.grid(alpha=0.3)

plt.suptitle("Sample anomaly-flagged sequences (model 3 training examples)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
life = pd.read_csv(FINDINGS_OUT)
sub = life.copy()
sub["bucket"] = sub.apply(lambda r: "shadow" if r["is_shadow"] else ("zombie" if r["is_zombie"] else r["lifecycle_state"]), axis=1)

exploded = life["owasp_findings"].fillna("").str.split("|").explode()
counts = exploded[exploded != ""].value_counts()

categories = [
    "API1:BOLA",
    "API2:Broken-Authentication",
    "API4:Unrestricted-Resource-Consumption",
    "API8:Security-Misconfiguration",
    "API9:Improper-Inventory-Management",
    "API10:Unsafe-Consumption-Of-APIs",
]
order = ["active", "deprecated", "orphaned", "zombie", "shadow"]

matrix = []
for bucket in order:
    rows = sub[sub["bucket"] == bucket]
    if len(rows) == 0:
        matrix.append([0.0] * len(categories))
        continue
    matrix.append([rows["owasp_findings"].fillna("").str.contains(c, regex=False).mean() for c in categories])
matrix = np.array(matrix)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

axes[0].barh(counts.index[::-1], counts.values[::-1], color="#d62728")
axes[0].set_xlabel("count")
axes[0].set_title("OWASP API Top 10 finding frequency")
for i, v in enumerate(counts.values[::-1]):
    axes[0].text(v + 10, i, str(v), va="center", fontsize=9)

im = axes[1].imshow(matrix, cmap="Reds", aspect="auto", vmin=0, vmax=1)
axes[1].set_xticks(range(len(categories)))
axes[1].set_xticklabels([c.split(":")[0] for c in categories])
axes[1].set_yticks(range(len(order)))
axes[1].set_yticklabels(order)
axes[1].set_title("Hit rate by sub-population (state × category)")
for i in range(len(order)):
    for j in range(len(categories)):
        axes[1].text(j, i, f"{matrix[i,j]:.0%}", ha="center", va="center",
                     color="white" if matrix[i, j] > 0.5 else "black", fontsize=9)
plt.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()